# 05. A/B 테스트 통계 분석

Cookie Cats 게이트 위치 변경 A/B 테스트의 통계적 유의성 검증

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import norm

from src.data.loader import load_cookie_cats

In [ ]:
df = load_cookie_cats()
print(f"Shape: {df.shape}")
print(f"\nVersion 분포:")
print(df["version"].value_counts())
df.head()

## 1. 리텐션 기본 분석

In [ ]:
retention = df.groupby("version")[["retention_1", "retention_7"]].mean().round(4)
print("그룹별 리텐션율:")
print(retention)

ret_df = retention.reset_index().melt(id_vars="version", var_name="metric", value_name="rate")
fig = px.bar(ret_df, x="metric", y="rate", color="version", barmode="group",
             title="A/B 그룹별 리텐션율", text_auto=".4f")
fig.show()

## 2. Z-test (이표본 비율 검정)

In [ ]:
def two_proportion_ztest(p1, n1, p2, n2):
    p_pool = (p1 * n1 + p2 * n2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p1 - p2) / se
    p_value = 2 * (1 - norm.cdf(abs(z)))
    return z, p_value

g30 = df[df["version"] == "gate_30"]
g40 = df[df["version"] == "gate_40"]

for metric in ["retention_1", "retention_7"]:
    p1, n1 = g30[metric].mean(), len(g30)
    p2, n2 = g40[metric].mean(), len(g40)
    z, p_val = two_proportion_ztest(p1, n1, p2, n2)
    sig = "유의" if p_val < 0.05 else "비유의"
    print(f"{metric}:")
    print(f"  gate_30: {p1:.4f}, gate_40: {p2:.4f}, 차이: {p1-p2:.4f}")
    print(f"  Z = {z:.4f}, p-value = {p_val:.6f} -> {sig}")
    print()

## 3. 효과 크기 (Cohen's h)

In [ ]:
def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

for metric in ["retention_1", "retention_7"]:
    p1 = g30[metric].mean()
    p2 = g40[metric].mean()
    h = cohens_h(p1, p2)
    if abs(h) < 0.2:
        size = "negligible"
    elif abs(h) < 0.5:
        size = "small"
    elif abs(h) < 0.8:
        size = "medium"
    else:
        size = "large"
    print(f"{metric}: Cohen's h = {h:.4f} ({size})")

## 4. 95% 신뢰구간

In [ ]:
for metric in ["retention_1", "retention_7"]:
    p1, n1 = g30[metric].mean(), len(g30)
    p2, n2 = g40[metric].mean(), len(g40)
    diff = p1 - p2
    se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    ci_low = diff - 1.96 * se
    ci_high = diff + 1.96 * se
    print(f"{metric}: 차이 = {diff:.4f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

metrics_data = []
for metric in ["retention_1", "retention_7"]:
    p1, n1 = g30[metric].mean(), len(g30)
    p2, n2 = g40[metric].mean(), len(g40)
    diff = p1 - p2
    se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    metrics_data.append({"metric": metric, "diff": diff, "ci_low": diff-1.96*se, "ci_high": diff+1.96*se})

ci_df = pd.DataFrame(metrics_data)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ci_df["diff"], y=ci_df["metric"],
    error_x=dict(type="data", symmetric=False,
                 array=(ci_df["ci_high"]-ci_df["diff"]).tolist(),
                 arrayminus=(ci_df["diff"]-ci_df["ci_low"]).tolist()),
    mode="markers", marker=dict(size=10)
))
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(title="리텐션 차이 95% 신뢰구간 (gate_30 - gate_40)", xaxis_title="차이")
fig.show()

## 5. 부트스트랩 시뮬레이션

In [ ]:
np.random.seed(42)
n_bootstrap = 10000
boot_diffs = []

for _ in range(n_bootstrap):
    s30 = g30["retention_7"].sample(n=len(g30), replace=True)
    s40 = g40["retention_7"].sample(n=len(g40), replace=True)
    boot_diffs.append(s30.mean() - s40.mean())

boot_diffs = np.array(boot_diffs)
ci_2_5 = np.percentile(boot_diffs, 2.5)
ci_97_5 = np.percentile(boot_diffs, 97.5)
observed = g30["retention_7"].mean() - g40["retention_7"].mean()

print(f"Bootstrap 95% CI: [{ci_2_5:.4f}, {ci_97_5:.4f}]")
print(f"관측된 차이: {observed:.4f}")

fig = px.histogram(x=boot_diffs, nbins=50, title="Bootstrap: retention_7 차이 분포")
fig.add_vline(x=observed, line_color="red", annotation_text="관측값")
fig.add_vline(x=ci_2_5, line_dash="dash", line_color="green", annotation_text="2.5%")
fig.add_vline(x=ci_97_5, line_dash="dash", line_color="green", annotation_text="97.5%")
fig.show()

## 6. 게임라운드별 리텐션

In [ ]:
df["rounds_bin"] = pd.cut(
    df["sum_gamerounds"],
    bins=[0, 1, 10, 50, 100, df["sum_gamerounds"].max()],
    labels=["0-1", "2-10", "11-50", "51-100", "100+"]
)

bin_ret = df.groupby(["rounds_bin", "version"])["retention_7"].mean().reset_index()
fig = px.bar(bin_ret, x="rounds_bin", y="retention_7", color="version", barmode="group",
             title="게임라운드 구간별 7일 리텐션", text_auto=".3f")
fig.show()

## 7. 결론 & 비즈니스 인사이트

### 주요 발견
- **gate_30 (현재)** 이 gate_40보다 리텐션이 높음
- retention_7에서 통계적으로 유의미한 차이 존재
- 효과 크기(Cohen's h)는 small 수준이나, 대규모 유저 기반에서는 비즈니스적으로 의미 있음

### 권장 사항
- 게이트를 level 30에 유지하는 것이 리텐션 관점에서 유리
- 게이트가 너무 늦으면 유저가 벽을 느끼기 전에 이탈할 수 있음
- 추가 실험: 게이트 위치를 20, 25 등으로 세분화 테스트 권장